Este notebook faz parte de uma avaliação do curso do SCTEC - Machine Learning e Visão Computacional 

Utilizei o dataset público Casting Product Image Data for Quality Inspection (https://drive.google.com/file/d/1K5gNxQ7RXA-nb4boNzPYQTJlRvJyYBD1/view?usp=sharing), que contém imagens reais de peças de fundição com e sem defeitos. 

Trata-se de um desafio técnico para desenvolver um pipeline (script) em Python utilizando a biblioteca OpenCV para realizar o pré-processamento de um lote de imagens de peças metálicas.
Ele prepara as imagens para que, a equipe de Machine Learning possa treinar um modelo preditivo. Aplico uma sequência lógica de filtros e transformações (como conversão para escala de cinza, suavização, limiarização, operações morfológicas e detecção de bordas) que evidenciem as ranhuras e defeitos estruturais das peças, salvando o resultado final de forma padronizada.

A Sprint 1 foi concluída, pois correspondeu à etapa de setup inicial do projeto.


Sprint 2
Uma estrutura de pastas (/raw_images e /processed_images) foi criada. Em /raw_images, estão armazenadas as imagens baixadas do dataset.

Inicio esta segunda parte da Sprint 2 para desenvolver a lógica para ler múltiplas imagens do diretório de entrada em lote (batch).

Sprint 3:
 Pipeline de Pré-processamento Base: Implementar conversão para escala de cinza (Grayscale) e 
 aplicação de filtro para redução de ruído (ex: Gaussian Blur ou Median Blur).

Sprint 4:
Segmentação e Destaque de Características: 
Aplicar técnicas de limiarização (Thresholding, como o método de Otsu) e detecção de bordas (ex: Canny ou Sobel) para destacar os contornos da peça e possíveis falhas

Sprint 5:
Refinamento Morfológico e Padronização: Utilizar operações morfológicas, especificamente o fechamento (Closing), baseado em dilatação seguida de erosão, para preservar e conectar as bordas das imagens e redimensioná-las para 256x256 pixels.

Sprint 6: 
Gravação e Documentação: Salvar as imagens processadas no diretório de saída,

In [ ]:
import cv2
from pathlib import Path
import time
from tqdm.auto import tqdm

class CastingPreprocessor:

    def __init__(self):
        pass

    def _read_images_batch(self, input_dir):
        """
        Lê todas as imagens da pasta e subpastas de entrada.
        Retorna uma lista com as imagens carregadas na memória.
        """

        input_dir = Path(input_dir)
        image_extensions = {".jpg", ".jpeg", ".png"}

        try:
            image_files = [
                file for file in input_dir.rglob("*")
                if file.is_file() and file.suffix.lower() in image_extensions
            ]

            if not image_files:
                raise FileNotFoundError(
                    "Nenhuma imagem foi encontrada no diretório de entrada."
                )

            images = []

            for file in image_files:
                image = cv2.imread(str(file))

                if image is not None:
                    images.append(image)

            print(f"Arquivos de imagem encontrados: {len(image_files)}")
            print(f"Imagens carregadas na memória: {len(images)}")

            if len(image_files) == len(images):
                print("Leitura das imagens realizada com sucesso.")
            else:
                print("Atenção: nem todas as imagens foram carregadas.")

            return images

        except FileNotFoundError as error:
            print(f"Erro: {error}")
            return []

        except Exception as error:
            print(f"Ocorreu um erro durante a leitura das imagens: {error}")
            return []

    def _to_grayscale(self, images):
        """
        Converte as imagens para escala de cinza.
        """

        try:
            gray_images = []

            for image in images:
                gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                gray_images.append(gray)

            return gray_images

        except Exception as error:
            print(f"Ocorreu um erro na conversão para escala de cinza: {error}")
            return []

    def _apply_blur(self, gray_images):
        """
        Aplica Gaussian Blur nas imagens em escala de cinza
        para redução de ruído.
        """

        try:
            blur_images = []

            for gray in gray_images:
                blur = cv2.GaussianBlur(gray, (5, 5), 0)
                blur_images.append(blur)

            return blur_images

        except Exception as error:
            print(f"Ocorreu um erro na aplicação do Gaussian Blur: {error}")
            return []

    def _apply_otsu(self, blur_images):
        """
        Aplica thresholding de Otsu nas imagens suavizadas.
        Retorna as imagens binárias e os valores de threshold.
        """

        try:
            threshold_images = []
            otsu_thresholds = []

            for blur in blur_images:
                otsu_threshold, binary = cv2.threshold(
                    blur,
                    0,
                    255,
                    cv2.THRESH_BINARY + cv2.THRESH_OTSU
                )

                threshold_images.append(binary)
                otsu_thresholds.append(otsu_threshold)

            return threshold_images, otsu_thresholds

        except Exception as error:
            print(f"Ocorreu um erro na aplicação do threshold de Otsu: {error}")
            return [], []

    def _apply_canny(self, threshold_images):
        """
        Aplica detecção de bordas Canny sobre as imagens resultantes do Otsu.
        """

        try:
            canny_images = []

            for binary in threshold_images:
                edges = cv2.Canny(binary, 50, 150)
                canny_images.append(edges)

            return canny_images

        except Exception as error:
            print(f"Ocorreu um erro na detecção de bordas Canny: {error}")
            return []

    def _apply_morphology_resize(self, canny_images, size=(256, 256)):
        """
        Redimensiona as imagens Canny para 256x256 pixels
        e aplica fechamento morfológico (Closing) com kernel 3x3.
        """

        try:
            processed_images = []

            kernel = cv2.getStructuringElement(
                cv2.MORPH_RECT,
                (3, 3)
            )

            for canny in canny_images:

                resized = cv2.resize(
                    canny,
                    size,
                    interpolation=cv2.INTER_AREA
                )

                closing = cv2.morphologyEx(
                    resized,
                    cv2.MORPH_CLOSE,
                    kernel
                )

                processed_images.append(closing)

            return processed_images

        except Exception as error:
            print(
                f"Ocorreu um erro na aplicação da morfologia e resize: {error}"
            )
            return []

    def _save_images(self, images, output_dir):
        """
        Salva as imagens processadas no diretório de saída.
        """

        try:
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)

            for i, image in enumerate(images, start=1):
                output_path = output_dir / f"processed_{i:02d}.png"
                cv2.imwrite(str(output_path), image)

            print(f"{len(images)} imagens salvas em: {output_dir}")

        except Exception as error:
            print(f"Ocorreu um erro ao salvar as imagens: {error}")


    def pipeline(self, input_dir, output_dir):
        """
        Executa o pipeline completo de pré-processamento
        das imagens em lote.
        """

        try:
            start_time = time.perf_counter()

            progress = tqdm(
                total=7,
                desc="Processando pipeline",
                unit=" etapa"
            )

            # 1. Leitura das imagens
            images = self._read_images_batch(input_dir)
            progress.update(1)

            if not images:
                progress.close()
                return

            # 2. Escala de cinza
            gray_images = self._to_grayscale(images)
            progress.update(1)

            if not gray_images:
                progress.close()
                return

            # 3. Redução de ruído
            blur_images = self._apply_blur(gray_images)
            progress.update(1)

            if not blur_images:
                progress.close()
                return

            # 4. Thresholding de Otsu
            threshold_images, otsu_thresholds = self._apply_otsu(
                blur_images
            )
            progress.update(1)

            if not threshold_images:
                progress.close()
                return

            # 5. Detecção de bordas Canny
            canny_images = self._apply_canny(threshold_images)
            progress.update(1)

            if not canny_images:
                progress.close()
                return

            # 6. Morfologia + Resize
            processed_images = self._apply_morphology_resize(
                canny_images
            )
            progress.update(1)

            if not processed_images:
                progress.close()
                return

            # 7. Salvamento
            self._save_images(
                processed_images,
                output_dir
            )
            progress.update(1)

            progress.close()

            elapsed_time = time.perf_counter() - start_time

            print()
            print("✅ Processamento concluído!")
            print(f"⏱️ Tempo total: {elapsed_time:.2f} segundos")

        except Exception as error:
            print(f"Ocorreu um erro no pipeline: {error}")

                

In [ ]:
processor = CastingPreprocessor()

In [ ]:
processor.pipeline(
    "raw_images",
    "processed_images"
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path


input_dir = Path("raw_images")
output_dir = Path("processed_images")

image_extensions = {".jpg", ".jpeg", ".png"}

# Usa a mesma ordem de leitura do pipeline
image_files = [
    file for file in input_dir.rglob("*")
    if file.is_file() and file.suffix.lower() in image_extensions
]

# Seleciona 2 imagens defeituosas e 2 imagens OK
defect_files = [
    file for file in image_files
    if "cast_def" in file.name.lower()
][:2]

ok_files = [
    file for file in image_files
    if "cast_ok" in file.name.lower()
][:2]

sample_files = defect_files + ok_files


# Cria a visualização
fig, axes = plt.subplots(
    len(sample_files), 2,
    figsize=(10, 14)
)

for row, file in enumerate(sample_files):

    # Imagem original
    original = cv2.imread(str(file))
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    # Descobre a posição da imagem original na lista do pipeline
    original_index = image_files.index(file)

    # Localiza exatamente a imagem processada correspondente
    processed_path = (
        output_dir / f"processed_{original_index + 1:02d}.png"
    )

    processed = cv2.imread(str(processed_path))
    processed = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

    # Original
    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f"Original\n{file.name}")
    axes[row, 0].axis("off")

    # Processada
    axes[row, 1].imshow(processed)
    axes[row, 1].set_title(
        f"Processada\n{processed_path.name}"
    )
    axes[row, 1].axis("off")


plt.suptitle(
    "Comparação: Imagem Original × Imagem Processada",
    fontsize=16
)

plt.tight_layout()
plt.show()